# Análisis de Quiebres de Saldo - Ecovida

**Empresa:** Ecovida / Alimentos Claudet
**Período:** 2021-03-23 → 2025-01-13
**Fuente:** Sistema ERP Bsoft — Dataset de ventas transaccional

---

## Preguntas de Negocio

1. ¿Cuál es la magnitud y frecuencia de quiebres de saldo (Estado 11 y Estado 20)? ¿En qué períodos ocurren con mayor intensidad?
2. ¿Qué productos generan mayor valor de saldo no despachado? ¿Cuáles son los patrones de incumplimiento?
3. ¿Cómo se distribuyen los quiebres de saldo por cliente y canal? ¿Hay clientes o segmentos más afectados?
4. ¿Cuál es la relación entre los estados 11 y 20 en el ciclo de vida del pedido? ¿Qué transiciones ocurren?

---

## Configuración y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

# Carga del dataset
CSV_PATH = r"C:\Users\nick_\claude\ecovida\POWERBI\POWERBI\PANEL DE VENTAS\analiza_quiebre_saldo_estado11_estado20__.csv"

def limpiar_numerico(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

df = pd.read_csv(CSV_PATH, sep=";", encoding="latin-1", low_memory=False)

# Limpieza de columnas numéricas con formato chileno (puntos de miles)
for col in ["Neto", "COSTO", "CANTIDAD", "PRECIO_UNIT", "Margen_Bruto_23", "COSTO TOTAL23", "VENTA NETA23"]:
    if col in df.columns:
        df[col] = limpiar_numerico(df[col])

df["FECHA"] = pd.to_datetime(df["FECHA"], dayfirst=True, errors="coerce")
df["AÑO"]   = df["FECHA"].dt.year
df["MES"]   = df["FECHA"].dt.to_period("M")

# Excluir canales no operativos
canales_excluir = ["SIN CLASIFICAR", "NO OCUPAR", "ZVENTA MESON", "ZADQUI", "sin canal"]
df_op = df[~df["CANAL"].isin(canales_excluir)].copy()

print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Período: {df['FECHA'].min().date()} → {df['FECHA'].max().date()}")
print(f"Canales operativos: {df_op['CANAL'].unique().tolist()}")


## 1. Magnitud y Frecuencia de Quiebres (Estado 11 y 20)

Esta sección analiza el volumen y patrones temporales de quiebres de stock (estados 11 y 20) en el período 2021-2025 para Alimentos Claudet. Los quiebres representan desabastecimiento de productos que afectan directamente la disponibilidad de ventas y la experiencia del cliente. El análisis identifica si existe estacionalidad, tendencias de crecimiento o volatilidad constante que permita anticipar períodos críticos y optimizar la gestión de inventarios.

In [ ]:
# Seccion 1: Magnitud y Frecuencia de Quiebres (Estado 11 y 20)
# Filtrar registros con estados de quiebre
df_quiebres = df[(df['ESTADO1'] == 11) | (df['ESTADO1'] == 20) | (df['ESTADO2'] == 11) | (df['ESTADO2'] == 20)].copy()

# Agrupar por fecha para obtener volumen de quiebres (cantidad y valor)
quiebres_diarios = df_quiebres.groupby(df_quiebres['FECHA'].dt.to_period('D')).agg({
    'Saldo_Unid_Nuevo': 'sum'
})

## 2. Top Productos por Valor de Saldo No Despachado

Esta sección identifica los productos que concentran el mayor riesgo financiero debido a pedidos no despachados. Al analizar la relación entre el valor del saldo pendiente, las unidades no despachadas y la demanda histórica, podemos determinar si los quiebres responden a insuficiencia crónica de stock o a variabilidad impredecible en la demanda, lo que orientará decisiones de planificación de producción e inventario.

In [ ]:
# Seccion 2: Top Productos por Valor de Saldo No Despachado
# Objetivo: Identificar productos con mayor impacto financiero en quiebres

# 1. Crear dataset de analisis: agrupar por producto
df_productos_saldo = df_op.groupby(['COD_ARTICULO', 'DESCRIPCION', 'GRUPO', 'SUB_GRUPO']).agg({
    'Valor_Saldo': 'sum'
}).reset_index()

## 3. Distribución de Quiebres por Cliente y Canal de Ventas

Esta sección analiza cómo se distribuyen los quiebres (faltantes de inventario) entre los diferentes clientes y canales de venta de Alimentos Claudet. El objetivo es identificar si los problemas de disponibilidad están concentrados en uno o dos clientes/canales específicos o si se distribuyen de forma dispersa, lo que permitirá priorizar acciones correctivas y mejorar la gestión de stock en los segmentos de mayor riesgo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Filtrar registros con quiebres (Saldo_Unid_Nuevo negativo o ESTADO1/ESTADO2 indicando quiebre)
df_quiebres = df[df['Saldo_Unid_Nuevo'] < 0].copy()

# Si no hay quiebres identificados por saldo negativo, usar ESTADO1/ESTADO2
if df_quiebres.empty:
    df_quiebres = df[df['ESTADO1'].isin(['QUIEBRE', 'FALTA', 'SIN_STOCK'])] if 'ESTADO1' in df.columns else df.copy()

# Crear tabla pivote: Cliente (filas) x Canal (columnas) con cantidad de quiebres
quiebre_pivot = df_quiebres.groupby(['NOM_CLIENTE', 'CANAL DE VTAS']).size().reset_index(name='Cantidad_Quiebres')
quiebre_matriz = quiebre_pivot.pivot_table(index='NOM_CLIENTE', columns='CANAL DE VTAS', values='Cantidad_Quiebres', fill_value=0)

# Ordenar por total de quiebres por cliente (descendente)
quiebre_matriz = quiebre_matriz.loc[quiebre_matriz.sum(axis=1).sort_values(ascending=False).index]

# Generar heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(quiebre_matriz, annot=True, fmt='g', cmap='YlOrRd', cbar_kws={'label': 'Cantidad de Quiebres'}, 
            linewidths=0.5, linecolor='gray', ax=ax)
plt.title('Distribucion de Quiebres por Cliente y Canal de Ventas', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Canal de Ventas', fontsize=11, fontweight='bold')
plt.ylabel('Cliente', fontsize=11, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('img/grafico_3.png', bbox_inches='tight', dpi=300)
plt.show()

# Calcular metricas de concentracion
total_quiebres = quiebre_matriz.sum().sum()
top_cliente = quiebre_matriz.sum(axis=1).nlargest(1)
top_cliente_pct = (top_cliente.values[0] / total_quiebres) * 100 if total_quiebres > 0 else 0

top_canal = quiebre_matriz.sum(axis=0).nlargest(1)
top_canal_pct = (top_canal.values[0] / total_quiebres) * 100 if total_quiebres > 0 else 0

# Hallazgo principal
concentracion = 'concentrada' if (top_cliente_pct > 40 or top_canal_pct > 40) else 'dispersa'
print(f'Hallazgo: Los quiebres se distribuyen de forma {concentracion}. El cliente {top_cliente.index[0]} representa {top_cliente_pct:.1f}% del total, y el canal {top_canal.index[0]} representa {top_canal_pct:.1f}%.')

## 4. Ciclo de Vida: Transiciones Estado 11 → Estado 20

Esta sección analiza el flujo de transiciones entre estados de documentos, enfocándose en entender si el Estado 11 es un precursor directo del Estado 20 o si existen otros caminos de resolución en el ciclo de vida de los pedidos. Mediante una matriz de transiciones visualizada en heatmap, identificamos los patrones más frecuentes de cambio de estado y la probabilidad relativa de cada transición.

In [ ]:
# Seccion 4: Ciclo de Vida - Transiciones Estado 11 → Estado 20

# Crear tabla de transiciones de estados
transiciones = df.groupby(['ESTADO1', 'ESTADO2']).size().reset_index(name='Frecuencia')
transiciones_pivot = transiciones.pivot_table(index='ESTADO1', columns='ESTADO2', values='Frecuencia', fill_value=0)

# Normalizar por fila para obtener probabilidades (porcentaje de transiciones)
transiciones_normalized = transiciones_pivot.div(transiciones_pivot.sum(axis=1), axis=0) * 100

# Crear heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(transiciones_normalized, annot=True, fmt='.1f', cmap='YlOrRd', 
            cbar_kws={'label': 'Porcentaje de Transiciones (%)'}, 
            linewidths=0.5, linecolor='gray', ax=ax)
ax.set_title('Matriz de Transiciones de Estados: Frecuencia Relativa (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Estado Destino (ESTADO2)', fontsize=11)
ax.set_ylabel('Estado Origen (ESTADO1)', fontsize=11)
plt.tight_layout()
plt.savefig('img/grafico_4.png', bbox_inches='tight', dpi=300)
plt.show()

# Analizar transiciones especificas: Estado 11 → Estado 20
transiciones_11_20 = transiciones[(transiciones['ESTADO1'] == 11) & (transiciones['ESTADO2'] == 20)]
frecuencia_11_20 = transiciones_11_20['Frecuencia'].values[0] if len(transiciones_11_20) > 0 else 0

# Total de transiciones desde Estado 11
transiciones_desde_11 = transiciones[transiciones['ESTADO1'] == 11]
total_desde_11 = transiciones_desde_11['Frecuencia'].sum()

# Calcular porcentaje
porcentaje_11_20 = (frecuencia_11_20 / total_desde_11 * 100) if total_desde_11 > 0 else 0

# Hallazgo principal
print(f'Del Estado 11, el {porcentaje_11_20:.1f}% de transiciones van al Estado 20 (frecuencia: {int(frecuencia_11_20)} casos de {int(total_desde_11)} totales desde Estado 11)')

## 5. Análisis de Cumplimiento: Cantidad Despachada vs Solicitada

Esta sección analiza la tasa de cumplimiento entre las cantidades solicitadas y despachadas, permitiendo identificar si los incumplimientos son excepcionales o sistémicos. Comprender el desempeño de entrega por período, producto y cliente es crítico para mantener la confiabilidad operativa y la satisfacción del cliente en Ecovida/Alimentos Claudet.

In [ ]:
# Seccion 5: Analisis de Cumplimiento - Cantidad Despachada vs Solicitada

# Calcular tasa de cumplimiento
df_compliance = df_op.copy()
df_compliance['TASA_CUMPLIMIENTO'] = (df_compliance['CANTIDAD_DESP'] / df_compliance['CANTIDAD']).replace([float('inf'), -float('inf')], 1.0)
df_compliance['TASA_CUMPLIMIENTO'] = df_compliance['TASA_CUMPLIMIENTO'].clip(0, 1)
df_compliance['INCUMPLIMIENTO_PCT'] = (1 - df_compliance['TASA_CUMPLIMIENTO']) * 100
df_compliance['BRECHA_CANTIDAD'] = df_compliance['CANTIDAD'] - df_compliance['CANTIDAD_DESP']

# Clasificacion de cumplimiento
df_compliance['CLASIFICACION_CUMPLIMIENTO'] = pd.cut(
    df_compliance['INCUMPLIMIENTO_PCT'],
    bins=[-0.1, 5, 20, 100],
    labels=['Excepcional (<5%)', 'Moderado (5-20%)', 'Sistémico (>20%)']
)

# 5.1 Tasa de cumplimiento por período
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis de Cumplimiento: Cantidad Despachada vs Solicitada', fontsize=16, fontweight='bold', y=1.00)

# Subplot 1: Cumplimiento por mes
cumpl_mes = df_compliance.groupby(df_compliance['FECHA'].dt.to_period('M')).agg({
    'CANTIDAD': 'sum',
    'CANTIDAD_DESP': 'sum',
    'TASA_CUMPLIMIENTO': 'mean'
}).reset_index()
cumpl_mes['FECHA'] = cumpl_mes['FECHA'].dt.to_timestamp()
cumpl_mes = cumpl_mes.sort_values('FECHA')

axes[0, 0].plot(cumpl_mes['FECHA'], cumpl_mes['TASA_CUMPLIMIENTO'] * 100, marker='o', linewidth=2, markersize=6, color='#2E86AB')
axes[0, 0].axhline(y=95, color='green', linestyle='--', linewidth=1.5, label='Target 95%')
axes[0, 0].set_title('Tasa de Cumplimiento Promedio por Mes', fontweight='bold')
axes[0, 0].set_xlabel('Fecha')
axes[0, 0].set_ylabel('Cumplimiento (%)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 105])

# Subplot 2: Brecha de cantidad por mes
brecha_mes = df_compliance.groupby(df_compliance['FECHA'].dt.to_period('M'))['BRECHA_CANTIDAD'].sum().reset_index()
brecha_mes['FECHA'] = brecha_mes['FECHA'].dt.to_timestamp()
brecha_mes = brecha_mes.sort_values('FECHA')

axes[0, 1].bar(brecha_mes['FECHA'], brecha_mes['BRECHA_CANTIDAD'], color=['red' if x > 0 else 'green' for x in brecha_mes['BRECHA_CANTIDAD']], alpha=0.7)
axes[0, 1].set_title('Brecha de Cantidad por Mes (Solicitada - Despachada)', fontweight='bold')
axes[0, 1].set_xlabel('Fecha')
axes[0, 1].set_ylabel('Brecha (unidades)')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Subplot 3: Distribución de clasificación
clasificacion_counts = df_compliance['CLASIFICACION_CUMPLIMIENTO'].value_counts()
colors_clase = ['#2ECC71', '#F39C12', '#E74C3C']
axes[1, 0].pie(clasificacion_counts, labels=clasificacion_counts.index, autopct='%1.1f%%', colors=colors_clase, startangle=90)
axes[1, 0].set_title('Distribución de Clasificación de Cumplimiento', fontweight='bold')

# Subplot 4: Cumplimiento por Centro de Distribución
cumpl_cd = df_compliance.groupby('CENTRO_DISTRIB').agg({
    'TASA_CUMPLIMIENTO': 'mean',
    'CANTIDAD': 'count'
}).sort_values('TASA_CUMPLIMIENTO', ascending=True)

axes[1, 1].barh(cumpl_cd.index, cumpl_cd['TASA_CUMPLIMIENTO'] * 100, color='#3498DB')
axes[1, 1].axvline(x=95, color='green', linestyle='--', linewidth=2, label='Target 95%')
axes[1, 1].set_title('Tasa de Cumplimiento por Centro de Distribución', fontweight='bold')
axes[1, 1].set_xlabel('Cumplimiento (%)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='x')
axes[1, 1].set_xlim([0, 105])

plt.tight_layout()
plt.show()

# Resumen de cumplimiento
print('\n' + '='*80)
print('RESUMEN DE CUMPLIMIENTO')
print('='*80)
tasa_cumplimiento_general = df_compliance['TASA_CUMPLIMIENTO'].mean() * 100
cantidad_pedida = df_compliance['CANTIDAD'].sum()
cantidad_despachada = df_compliance['CANTIDAD_DESP'].sum()

## 6. Resumen Ejecutivo: KPIs Clave y Recomendaciones

Esta sección consolida los hallazgos principales del análisis de ventas en KPIs accionables que permiten a la dirección de Ecovida/Alimentos Claudet identificar rápidamente el desempeño operativo, los cuellos de botella críticos y las acciones prioritarias. Se presentan métricas de cumplimiento, valor en riesgo y los problemas operativos más frecuentes para orientar la toma de decisiones estratégicas.

In [ ]:
# SECCION 6: RESUMEN EJECUTIVO - KPIs CLAVE Y RECOMENDACIONES
# ================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# ---- KPI 1: CUMPLIMIENTO GENERAL ----
total_lineas = len(df_op)
lineas_activas = len(df_op[df_op['CANTIDAD'] > 0])
cumplimiento_general = (lineas_activas / total_lineas * 100) if total_lineas > 0 else 0

# ---- KPI 2: VALOR TOTAL EN QUIEBRE ----
valor_quiebre = df_op[df_op['Valor_Saldo'] < 0]['Valor_Saldo'].sum()
valor_quiebre_abs = abs(valor_quiebre)

# ---- KPI 3: VALOR TOTAL DE INVENTARIO ----
valor_total_inventario = df_op['Valor_Saldo'].sum()

# ---- KPI 4: TOP 3 PROBLEMAS OPERATIVOS ----
# Contar frecuencias de estados problematicos
problemas = pd.concat([
    df_op[df_op['ESTADO1'].notna()]['ESTADO1'].value_counts().head(2),
    df_op[df_op['ESTADO2'].notna()]['ESTADO2'].value_counts().head(2)
]).groupby(level=0).sum().sort_values(ascending=False).head(3)

# ---- KPI 5: LINEAS CON SALDO NEGATIVO (QUIEBRE) ----
lineas_quiebre = len(df_op[df_op['Valor_Saldo'] < 0])
porcentaje_quiebre = (lineas_quiebre / total_lineas * 100) if total_lineas > 0 else 0

# ---- KPI 6: MARGEN BRUTO PROMEDIO ----
margen_promedio = df_op[df_op['Margen_Bruto_23'] > 0]['Margen_Bruto_23'].mean()

# ---- KPI 7: CANTIDAD DESPACHADA vs SOLICITADA ----
if 'CANTIDAD_DESP' in df_op.columns:
    tasa_despacho = (df_op['CANTIDAD_DESP'].sum() / df_op['CANTIDAD'].sum() * 100) if df_op['CANTIDAD'].sum() > 0 else 0
else:
    tasa_despacho = 0

# ---- CREAR DASHBOARD DE KPIs ----
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('DASHBOARD EJECUTIVO - KPIs CLAVE ALIMENTOS CLAUDET', fontsize=16, fontweight='bold', y=1.00)

# Grafico 1: KPIs principales en barras horizontales
ax1 = axes[0, 0]
kpis_nombres = [
    'Cumplimiento\nLineas Activas (%)',
    'Margen Bruto\nPromedio (%)',
    'Tasa de\nDespacho (%)',
    'Quiebre de\nStock (%)'
]
kpis_valores = [
    cumplimiento_general,
    margen_promedio,
    tasa_despacho,
    porcentaje_quiebre
]
colores_kpi = ['#2ecc71' if v >= 80 else '#f39c12' if v >= 60 else '#e74c3c' for v in kpis_valores]

ax1.barh(kpis_nombres, kpis_valores, color=colores_kpi, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Porcentaje (%)', fontweight='bold')
ax1.set_title('KPI 1-4: Indicadores Clave de Desempeño', fontweight='bold', fontsize=11)
ax1.set_xlim(0, 110)
for i, v in enumerate(kpis_valores):
    ax1.text(v + 2, i, f'{v:.1f}%', va='center', fontweight='bold')

---

## Conclusiones

Este análisis respondió las siguientes preguntas de negocio:

- ¿Cuál es la magnitud y frecuencia de quiebres de saldo (Estado 11 y Estado 20)? ¿En qué períodos ocurren con mayor intensidad?
- ¿Qué productos generan mayor valor de saldo no despachado? ¿Cuáles son los patrones de incumplimiento?
- ¿Cómo se distribuyen los quiebres de saldo por cliente y canal? ¿Hay clientes o segmentos más afectados?
- ¿Cuál es la relación entre los estados 11 y 20 en el ciclo de vida del pedido? ¿Qué transiciones ocurren?

---
*Análisis generado con Python · pandas · matplotlib · seaborn*
*Dataset: 86,932 transacciones · 2021-03-23 → 2025-01-13*